# CS781 Phase 6 — Experiment 1
## Model 4 (Efficient Decoder) + LoRMA+ (Low-Rank Multiplicative Adaptation)

This notebook adapts the **existing Phase 1 Model 4 checkpoint** for legal abstractive summarization. It does not pretrain a new language model and does not load external pretrained weights.

**Method:** LoRMA+ on selected decoder projection layers. The original checkpoint weights are frozen; trainable low-rank matrices learn a multiplicative transformation of each selected layer's output.

**Before running:** set the paths in the configuration cell (or leave them as `None` to auto-discover unique files under `/kaggle/input`). The architecture is rebuilt from the checkpoint's own `config`, so do not edit model dimensions by hand.

**Data:** only the official Phase 6 `train.json` and `test.json` are used. A document-grouped held-out validation split avoids duplicate-document leakage. Long documents are cut to a bounded window (opening 2/3 + closing 1/3) for this first controlled experiment; hierarchical/chunked input is left for a later experiment.

**Run order:** run cells top to bottom. Training checkpoints contain only the small adapter weights, and `RESUME = True` continues an interrupted run.

**Outputs:** validation metrics using the embedded course evaluator and evaluator-ready JSON files; resumable test predictions; and a verified `phase6_exp1_submission.zip` with `prediction.json` at the archive root.

**Dataset access:** Phase 6 train/test JSON files are fetched from `Exploration-Lab/CS-781-Capstone` under `Phase_6_Summ` using the Hugging Face Hub. For a gated/private repository, add a Kaggle Secret named `HF_TOKEN`.

In [12]:
# 0. Environment and imports
import os, sys, json, math, random, time, gc, hashlib, re, subprocess, importlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 2026

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"
if USE_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # bf16 only where it is natively supported (Ampere+); T4/P100 use fp16 + GradScaler.
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float16   # unused: autocast is disabled on CPU
USE_GRAD_SCALER = USE_CUDA and AMP_DTYPE == torch.float16
torch.set_float32_matmul_precision("high")

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_CUDA)

def ensure_package(import_name, pip_name=None):
    # Import a package, pip-installing it first if it is missing.
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if USE_CUDA else "CPU",
      "| AMP:", str(AMP_DTYPE).replace("torch.", "") if USE_CUDA else "off")

torch 2.11.0+cu128 | device: cuda | Tesla T4 | AMP: float16


In [13]:
# 1. CONFIGURATION — EDIT THESE
# Leave a path as None to auto-discover it (the filename must be unique under the search roots).
CHECKPOINT_PATH = None   # e.g. "/kaggle/input/phase1-model4/eff_full_best.pt"
TOKENIZER_PATH  = None   # e.g. "/kaggle/input/phase1-model4/tokenizer.json"
TRAIN_JSON_PATH = None   # e.g. "/kaggle/working/phase6_data/train.json"
TEST_JSON_PATH  = None   # e.g. "/kaggle/working/phase6_data/test.json"

# Hugging Face Phase 6 dataset (shown in your screenshot)
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DATASET_SUBDIR = "Phase_6_Summ"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"  # Add this name under Kaggle Secrets if the repo is gated/private
CHECKPOINT_FILENAME = "eff_full_best (6).pt"   # used only for auto-discovery

ON_KAGGLE = Path("/kaggle/input").exists()
OUTPUT_DIR = Path("/kaggle/working/phase6_exp1_model4_lorma") if ON_KAGGLE else Path("./phase6_exp1_model4_lorma")
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]

# Data / sequence budget
VAL_FRACTION = 0.05
MAX_SOURCE_TOKENS = 1500
MAX_TARGET_TOKENS = 700
MAX_SEQ_LEN = 2200            # must be <= checkpoint max_positions

# Optimisation (conservative defaults for a ~99M model on a T4-class GPU)
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
LOG_EVERY = 50                # optimizer steps
EVAL_EVERY = 500              # optimizer steps (validation loss + best-checkpoint update)
SAVE_EVERY = 500              # optimizer steps (resumable "latest" checkpoint)
EVAL_MAX_EXAMPLES = 200       # fixed random subset of validation used for loss monitoring
RESUME = False                # True: continue training from the latest adapter checkpoint
RESUME_TEST_PREDICTIONS = True  # True: continue test generation from submission_partial.jsonl

# LoRMA+
LORMA_RANK = 8
LORMA_ALPHA = 16
LORMA_DROPOUT = 0.05
LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

# Generation / evaluation
GEN_MAX_NEW_TOKENS = 700
NO_REPEAT_NGRAM_SIZE = 0      # 0 = plain greedy. Try 10-16 if validation outputs loop (tokens are sub-word BPE).
VAL_SCORE_EXAMPLES = 100
BERTSCORE_BATCH_SIZE = 8
RUN_TEST_INFERENCE = True

# Fail fast on inconsistent settings
assert MICRO_BATCH_SIZE >= 1 and GRAD_ACCUM_STEPS >= 1 and NUM_EPOCHS >= 1
assert 0.0 < VAL_FRACTION < 0.5 and 0.0 <= WARMUP_RATIO < 1.0
assert 0 <= NO_REPEAT_NGRAM_SIZE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR.resolve())

Output: /kaggle/working/phase6_exp1_model4_lorma


In [14]:
# 2. Locate model files and fetch Phase 6 data from Hugging Face
def find_unique(filename, explicit=None):
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    hits = {}
    for root in SEARCH_ROOTS:
        if root.exists():
            for x in root.rglob(filename):
                if x.is_file():
                    hits[x.resolve()] = x
    hits = list(hits.values())
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename}. Set its explicit path in the configuration cell."
        )
    if len(hits) > 1:
        print("Multiple candidates for", filename)
        for x in hits[:20]:
            print("  ", x)
        raise RuntimeError(f"Set an explicit path for {filename} in the configuration cell.")
    return hits[0]

def get_hf_token():
    # Kaggle Secrets is optional for public datasets, but required for gated/private access.
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
        except Exception as exc:
            print(
                f"Could not read Kaggle secret '{HF_TOKEN_SECRET_NAME}'. "
                "Continuing without a token; this works only if the dataset is public."
            )
    return None

def download_phase6_jsons():
    try:
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import HfApi, hf_hub_download

    token = get_hf_token()
    api = HfApi(token=token)
    repo_files = api.list_repo_files(
        repo_id=HF_DATASET_REPO, repo_type="dataset", token=token
    )
    print(f"Found {len(repo_files)} files in dataset repository {HF_DATASET_REPO}")

    def choose_file(kind):
        # Accommodate the repository's Train/Test folders and minor filename variations.
        candidates = [
            name for name in repo_files
            if name.lower().endswith(".json")
            and HF_DATASET_SUBDIR.lower() in name.lower()
            and kind.lower() in [part.lower() for part in name.replace("\\", "/").split("/")]
        ]
        # Prefer exact conventional names if multiple JSONs exist.
        preferred = [name for name in candidates
                     if Path(name).name.lower() == f"{kind.lower()}.json"]
        candidates = preferred or candidates
        if len(candidates) != 1:
            print(f"Candidate {kind} JSON files:")
            for name in candidates:
                print("  ", name)
            raise FileNotFoundError(
                f"Expected exactly one {kind} JSON under {HF_DATASET_SUBDIR}; "
                "inspect the listed candidates and set TRAIN_JSON_PATH/TEST_JSON_PATH explicitly."
            )
        return candidates[0]

    train_file = choose_file("Train")
    test_file = choose_file("Test")
    print("HF train file:", train_file)
    print("HF test file :", test_file)

    train_local = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=train_file, repo_type="dataset", token=token
    )
    test_local = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=test_file, repo_type="dataset", token=token
    )
    return Path(train_local), Path(test_local)

# The Phase 1 checkpoint and tokenizer are separate from this HF dataset.
CHECKPOINT_PATH = find_unique(CHECKPOINT_FILENAME, CHECKPOINT_PATH)
TOKENIZER_PATH = find_unique("tokenizer.json", TOKENIZER_PATH)

if TRAIN_JSON_PATH is None or TEST_JSON_PATH is None:
    hf_train_path, hf_test_path = download_phase6_jsons()
    TRAIN_JSON_PATH = Path(TRAIN_JSON_PATH) if TRAIN_JSON_PATH else hf_train_path
    TEST_JSON_PATH = Path(TEST_JSON_PATH) if TEST_JSON_PATH else hf_test_path
else:
    TRAIN_JSON_PATH = find_unique("train.json", TRAIN_JSON_PATH)
    TEST_JSON_PATH = find_unique("test.json", TEST_JSON_PATH)

for label, pth in [("Checkpoint", CHECKPOINT_PATH), ("Tokenizer ", TOKENIZER_PATH),
                   ("Train     ", TRAIN_JSON_PATH), ("Test      ", TEST_JSON_PATH)]:
    if not Path(pth).is_file():
        raise FileNotFoundError(f"{label.strip()} file does not exist: {pth}")
    print(f"{label}: {pth}")


Found 34 files in dataset repository Exploration-Lab/CS-781-Capstone
HF train file: Phase_6_Summ/Train/train.json
HF test file : Phase_6_Summ/Test/test.json
Checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
Tokenizer : /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
Train     : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json
Test      : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json


In [15]:
# 3. Phase 1 tokenizer (byte-level BPE; must behave exactly like the Phase 1 implementation)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
_PRETOKEN_RE = re.compile(r"\S+|\s+")   # words and whitespace runs are BPE'd independently

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    pad_token_id = property(lambda self: self.stoi["<pad>"])
    unk_token_id = property(lambda self: self.stoi["<unk>"])
    bos_token_id = property(lambda self: self.stoi["<s>"])
    eos_token_id = property(lambda self: self.stoi["</s>"])

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l, r): i for i, (l, r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    @classmethod
    def load_json(cls, path):
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)
        tok.symbol_bytes = {f"{b:02x}": bytes([b]) for b in range(256)}
        for l, r in tok.merges:
            tok.symbol_bytes[l + r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]
        tok._reset_encode_cache()
        return tok

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()
        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)
        symbols = self.word_to_symbols(word)
        ranks = self._ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for j in range(len(symbols) - 1):
                r = ranks.get((symbols[j], symbols[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (symbols[j], symbols[j + 1]), r
            if best is None:
                break
            left, right = best
            merged, j = [], 0
            while j < len(symbols):
                if j + 1 < len(symbols) and symbols[j] == left and symbols[j + 1] == right:
                    merged.append(left + right); j += 2
                else:
                    merged.append(symbols[j]); j += 1
            symbols = merged
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        unk = self.unk_token_id
        out = []
        for unit in _PRETOKEN_RE.findall(text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok not in BPE_SPECIALS:
                buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
PAD_ID, BOS_ID, EOS_ID = tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id
print("Tokenizer vocab:", len(tokenizer))

# Round-trip must be lossless for plain text, newlines, tabs, and non-ASCII characters.
for probe in ["IN THE SUPREME COURT OF INDIA — Section 14.",
              "Line one.\n\nLine two\twith tab,  double  space.",
              "Appeal No. 123/2019 (धारा 302) — decided on 5th May, 2019."]:
    assert tokenizer.decode(tokenizer.encode(probe)) == probe, f"round-trip failed: {probe!r}"
print("Tokenizer round-trip OK")

Tokenizer vocab: 24576
Tokenizer round-trip OK


In [16]:
# 4. Model 4 architecture (GQA + RoPE + RMSNorm + SwiGLU). Parameter names match the Phase 1 checkpoint;
#    a KV-cache path is added for fast generation (it does not change any parameter or the maths).
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return x32.to(dtype) * self.weight

class RotaryTable(nn.Module):
    def __init__(self, head_dim, max_positions, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv)
        self.register_buffer("cos_cached", freqs.cos(), persistent=False)
        self.register_buffer("sin_cached", freqs.sin(), persistent=False)
    def forward(self, position_ids):
        return self.cos_cached[position_ids].unsqueeze(1), self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x, cos, sin):
    B, H, T, D = x.shape
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    xp = x.view(B, H, T, D // 2, 2)
    x0, x1 = xp[..., 0], xp[..., 1]
    return torch.stack((x0 * cos - x1 * sin, x0 * sin + x1 * cos), dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // n_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape
        H, KV, D = self.n_heads, self.n_kv_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, T, KV, D).transpose(1, 2)
        v = self.v_proj(x).view(B, T, KV, D).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k, v = torch.cat([pk, k], 2), torch.cat([pv, v], 2)
        new_kv = (k, v) if use_cache else None
        S = k.size(2)
        if self.kv_repeat != 1:
            ka = k.repeat_interleave(self.kv_repeat, dim=1)
            va = v.repeat_interleave(self.kv_repeat, dim=1)
        else:
            ka, va = k, v
        attn_mask, is_causal = None, False
        if T == 1:
            if key_valid is not None:
                attn_mask = key_valid[:, None, None, :]
        elif key_valid is None and past_len == 0:
            is_causal = True
        else:
            qi = torch.arange(T, device=x.device)[:, None] + past_len
            kj = torch.arange(S, device=x.device)[None, :]
            attn_mask = (kj <= qi)[None, None, :, :]
            if key_valid is not None:
                attn_mask = attn_mask & key_valid[:, None, None, :]
                if past_len == 0:
                    attn_mask = attn_mask | (kj == qi)[None, None, :, :]
        y = F.scaled_dot_product_attention(q, ka, va, attn_mask=attn_mask,
                                          dropout_p=self.dropout if self.training else 0.0,
                                          is_causal=is_causal)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y), new_kv

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, dropout, rms_eps):
        super().__init__()
        self.norm1 = RMSNorm(d_model, rms_eps)
        self.attn = GQACausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model, rms_eps)
        self.ff = SwiGLU(d_model, ffn_dim, dropout)
        self.resid_drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past_kv=None, use_cache=False):
        a, new_kv = self.attn(self.norm1(x), cos, sin, past_kv=past_kv, use_cache=use_cache)
        x = x + self.resid_drop(a)
        x = x + self.ff(self.norm2(x))
        return x, new_kv

class EfficientDecoderLM(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = cfg["d_model"]
        self.max_positions = cfg["max_positions"]
        dropout = cfg.get("dropout", 0.0)
        self.token_emb = nn.Embedding(vocab_size, self.d_model)
        self.drop = nn.Dropout(dropout)
        self.rope = RotaryTable(self.d_model // cfg["n_heads"], self.max_positions, cfg["rope_base"])
        self.blocks = nn.ModuleList([
            EfficientDecoderBlock(self.d_model, cfg["n_heads"], cfg["n_kv_heads"],
                                  cfg["ffn_dim"], dropout, cfg["rms_eps"])
            for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(self.d_model, cfg["rms_eps"])

    def forward_hidden(self, input_ids, past_kvs=None, use_cache=False):
        # Returns hidden states, or (hidden, new_kvs) when use_cache=True.
        B, T = input_ids.shape
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"sequence {past_len + T} > max_positions {self.max_positions}")
        pos = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)
        cos, sin = self.rope(pos)
        x = self.drop(self.token_emb(input_ids))
        new_kvs = []
        for i, block in enumerate(self.blocks):
            x, kv = block(x, cos, sin, None if past_kvs is None else past_kvs[i], use_cache)
            new_kvs.append(kv)
        h = self.final_norm(x)
        return (h, new_kvs) if use_cache else h

    def forward(self, input_ids):
        return F.linear(self.forward_hidden(input_ids), self.token_emb.weight)   # tied embeddings

In [17]:
# 5. Load the Phase 1 checkpoint and verify architecture / weight compatibility
def load_torch(path):
    # Prefer the safe tensor-only loader; fall back to full unpickling only for files you trust.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:
        print(f"weights_only load failed ({type(e).__name__}); falling back to full unpickling for {Path(path).name}")
        return torch.load(path, map_location="cpu", weights_only=False)

ckpt = load_torch(CHECKPOINT_PATH)
if "model" not in ckpt or "config" not in ckpt:
    raise ValueError("Expected a Model 4 checkpoint with 'model' and 'config' entries.")
cfg = dict(ckpt["config"])

# Phase 1 checkpoints may omit the RMSNorm epsilon or store it under another key.
# Normalize common aliases first; if none is present, use the RMSNorm class default
# (1e-5) used by this reconstructed Model 4 implementation.
if "rms_eps" not in cfg:
    for eps_key in ("rms_norm_eps", "norm_eps", "layer_norm_epsilon", "eps"):
        if eps_key in cfg:
            cfg["rms_eps"] = float(cfg[eps_key])
            print(f"Using checkpoint RMSNorm epsilon from config['{eps_key}'] = {cfg['rms_eps']}")
            break
    else:
        cfg["rms_eps"] = 1e-5
        print(
            "WARNING: checkpoint config has no RMSNorm epsilon field. "
            "Using rms_eps=1e-5 (the RMSNorm implementation default). "
            "If your original Phase 1 Model 4 used a different epsilon, set cfg['rms_eps'] "
            "to that exact value before constructing the model."
        )

REQUIRED_CFG = ["d_model", "n_layers", "n_heads", "n_kv_heads", "ffn_dim",
                "max_positions", "rope_base", "rms_eps"]
missing_cfg = [k for k in REQUIRED_CFG if k not in cfg]
if missing_cfg:
    raise KeyError(f"Checkpoint config missing: {missing_cfg}")
if int(cfg.get("vocab_size", len(tokenizer))) != len(tokenizer):
    raise ValueError(f"Tokenizer/checkpoint vocab mismatch: {len(tokenizer)} vs {cfg.get('vocab_size')}")
if MAX_SEQ_LEN > int(cfg["max_positions"]):
    raise ValueError(f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds checkpoint max_positions={cfg['max_positions']}")

model = EfficientDecoderLM(cfg, len(tokenizer))
model.load_state_dict(ckpt["model"], strict=True)     # raises on any missing/unexpected key
model.to(DEVICE).eval()
del ckpt; gc.collect()
print(f"Loaded Model 4: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("Architecture:", {k: cfg[k] for k in REQUIRED_CFG}, "| dropout:", cfg.get("dropout", 0.0))

Loaded Model 4: 99.3M parameters
Architecture: {'d_model': 640, 'n_layers': 17, 'n_heads': 10, 'n_kv_heads': 2, 'ffn_dim': 2048, 'max_positions': 2560, 'rope_base': 10000.0, 'rms_eps': 1e-05} | dropout: 0.0


In [18]:
# 6. Attach LoRMA+ adapters to the chosen Linear projections
# LoRMA+ transformation: h = (I + alpha/r * B A) W0 x.  Efficient form: y = W0 x;  out = y + alpha/r * B(A(y)).
# B is zero-initialised, so the adapted model starts exactly equal to the frozen Phase 1 model.
class LoRMAPlusLinear(nn.Module):
    def __init__(self, base: nn.Linear, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        dev, dt = base.weight.device, base.weight.dtype      # adapters live where the base layer lives
        self.rank, self.scale = rank, alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.A = nn.Linear(base.out_features, rank, bias=False, device=dev, dtype=dt)
        self.B = nn.Linear(rank, base.out_features, bias=False, device=dev, dtype=dt)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.B(self.A(self.dropout(y)))

def attach_lorma(module, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        full = f"{prefix}{name}"
        if isinstance(child, nn.Linear) and name in LORMA_TARGETS:
            setattr(module, name, LoRMAPlusLinear(child, LORMA_RANK, LORMA_ALPHA, LORMA_DROPOUT))
            replaced.append(full)
        else:
            replaced.extend(attach_lorma(child, full + "."))
    return replaced

def is_adapter_key(k):
    return k.endswith(".A.weight") or k.endswith(".B.weight")

def lorma_state_dict(m):
    return {k: v.detach().cpu().clone() for k, v in m.state_dict().items() if is_adapter_key(k)}

def load_lorma_state_dict(m, sd):
    expected = {k for k in m.state_dict() if is_adapter_key(k)}
    if set(sd) != expected:
        raise RuntimeError("Adapter keys do not match the model "
                           f"(missing={sorted(expected - set(sd))[:3]}, unexpected={sorted(set(sd) - expected)[:3]}). "
                           "Check LORMA_RANK / LORMA_TARGETS.")
    m.load_state_dict(sd, strict=False)   # base weights are intentionally absent from `sd`

# Reference logits from the untouched model, to prove the identity-at-initialisation property.
probe_ids = (torch.arange(64, device=DEVICE) % (len(tokenizer) - 4) + 4)[None, :]
model.eval()
with torch.no_grad():
    ref_logits = model(probe_ids).float()

model.requires_grad_(False)
replaced = attach_lorma(model)
expected_n = cfg["n_layers"] * len(LORMA_TARGETS)
assert len(replaced) == expected_n, f"expected {expected_n} adapted layers, found {len(replaced)}"
for name, p in model.named_parameters():
    p.requires_grad = is_adapter_key(name)
model.to(DEVICE)

with torch.no_grad():
    max_diff = (model(probe_ids).float() - ref_logits).abs().max().item()
assert max_diff < 1e-4, f"LoRMA+ is not identity at init (max diff {max_diff})"
del ref_logits

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"LoRMA+ adapted layers: {len(replaced)} | identity check max|diff|={max_diff:.2e}")
print(f"Trainable parameters: {n_trainable:,} ({100 * n_trainable / n_total:.3f}% of {n_total:,})")
assert n_trainable > 0

LoRMA+ adapted layers: 119 | identity check max|diff|=0.00e+00
Trainable parameters: 1,705,984 (1.689% of 101,015,424)


In [19]:
# 7. Load Phase 6 data and build a duplicate-grouped train/validation split
def load_records(path):
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):                      # id-keyed JSON
        return [{"id": str(k), **v} for k, v in obj.items() if isinstance(v, dict)]
    if not isinstance(obj, list):
        raise TypeError("Expected a JSON list or an ID-keyed object")
    return obj

_WS_RE = re.compile(r"\s+")
def doc_group_key(text):
    # Normalised-document hash: exact duplicate documents always land in the same split.
    return hashlib.sha1(_WS_RE.sub(" ", text).strip().lower().encode("utf-8")).hexdigest()

train_records = load_records(TRAIN_JSON_PATH)
test_records = load_records(TEST_JSON_PATH)

clean = []
for r in train_records:
    rid, doc, summ = str(r.get("id", "")), r.get("document", ""), r.get("summary", "")
    if rid and isinstance(doc, str) and doc.strip() and isinstance(summ, str) and summ.strip():
        clean.append({"id": rid, "document": doc.strip(), "summary": summ.strip()})
for r in clean:
    r["group"] = doc_group_key(r["document"])
print(f"Raw train: {len(train_records)} | usable: {len(clean)} | dropped: {len(train_records) - len(clean)} | test: {len(test_records)}")
assert len({r["id"] for r in clean}) == len(clean), "duplicate ids in train.json"

# Validate the test file up-front so inference cannot fail halfway through.
for i, r in enumerate(test_records):
    if not str(r.get("id", "")) or not isinstance(r.get("document"), str):
        raise ValueError(f"Invalid test record at index {i}")
assert len({str(r["id"]) for r in test_records}) == len(test_records), "duplicate ids in test.json"

groups = defaultdict(list)
for r in clean:
    groups[r["group"]].append(r)
group_keys = list(groups)
random.Random(SEED).shuffle(group_keys)
target_val = max(1, int(len(clean) * VAL_FRACTION))
val_keys, n_val = set(), 0
for key in group_keys:
    if n_val >= target_val:
        break
    val_keys.add(key); n_val += len(groups[key])

train_rows = [r for r in clean if r["group"] not in val_keys]
val_rows = [r for r in clean if r["group"] in val_keys]
random.Random(SEED + 1).shuffle(val_rows)     # fixed random order -> "first N" is a random sample
assert train_rows and val_rows
assert {r["id"] for r in train_rows}.isdisjoint(r["id"] for r in val_rows)
assert {r["group"] for r in train_rows}.isdisjoint(r["group"] for r in val_rows)

train_groups = {r["group"] for r in train_rows}
test_overlap = sum(doc_group_key(r["document"].strip()) in train_groups for r in test_records)
print(f"Train={len(train_rows)} | validation={len(val_rows)} | unique documents={len(groups)} "
      f"| duplicated-doc rows={len(clean) - len(groups)}")
print(f"Test documents that also appear (exactly) in train: {test_overlap}")

Raw train: 7030 | usable: 7028 | dropped: 2 | test: 100
Train=6677 | validation=351 | unique documents=7001 | duplicated-doc rows=27
Test documents that also appear (exactly) in train: 0


In [20]:
# 8. Build causal-LM examples:  <s> + instruction + source window + "SUMMARY:" separator + summary + </s>
# Only the summary (and EOS) is supervised. The prompt/separator are never truncated: the *source*
# window shrinks first when prompt + source + target would exceed MAX_SEQ_LEN.
PROMPT_TEMPLATE = "Summarize the following Indian court judgment accurately. Preserve the key facts, legal issues, reasoning, decision, and important citations. Do not invent facts.\n\nJUDGMENT:\n"
PROMPT_IDS = tokenizer.encode(PROMPT_TEMPLATE)
SEP_IDS = tokenizer.encode("\n\nSUMMARY:\n")
PREFIX_OVERHEAD = 1 + len(PROMPT_IDS) + len(SEP_IDS)       # BOS + instruction + separator
min_source_budget = MAX_SEQ_LEN - PREFIX_OVERHEAD - MAX_TARGET_TOKENS - 1
if min_source_budget < 256:
    raise ValueError(f"MAX_SEQ_LEN too small: only {min_source_budget} source tokens left for the longest target.")

def head_tail(ids, budget):
    # Keep the opening 2/3 and the closing 1/3 of the budget (facts early, decision late).
    # Nested property: head_tail(head_tail(x, B1), B2) == head_tail(x, B2) whenever B2 <= B1.
    assert budget >= 1
    if len(ids) <= budget:
        return ids
    left = budget * 2 // 3
    right = budget - left
    return ids[:left] + ids[-right:]

def build_prompt_ids(src_ids, budget):
    return [BOS_ID] + PROMPT_IDS + head_tail(src_ids, budget) + SEP_IDS

def tokenize_rows(rows, name):
    examples, n_trunc, t0 = [], 0, time.time()
    for i, r in enumerate(rows):
        full = tokenizer.encode(r["document"])
        n_trunc += len(full) > MAX_SOURCE_TOKENS
        src = head_tail(full, MAX_SOURCE_TOKENS)
        tgt = tokenizer.encode(r["summary"])[:MAX_TARGET_TOKENS]
        budget = min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN - PREFIX_OVERHEAD - len(tgt) - 1)
        prefix = build_prompt_ids(src, budget)
        ids = prefix + tgt + [EOS_ID]
        assert len(ids) <= MAX_SEQ_LEN
        examples.append((np.asarray(ids, dtype=np.int32), len(prefix)))
        if (i + 1) % 500 == 0:
            print(f"  [{name}] tokenized {i + 1}/{len(rows)} ({time.time() - t0:.0f}s)", flush=True)
    print(f"{name}: {len(examples)} examples | source truncated: {n_trunc} ({100 * n_trunc / max(1, len(rows)):.1f}%)")
    return examples

print("Tokenizing (pure-Python BPE; this can take a few minutes) ...")
train_examples = tokenize_rows(train_rows, "train")
val_examples = tokenize_rows(val_rows, "val")

class SummarizationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        ids, prefix_len = self.examples[i]
        ids_t = torch.from_numpy(ids.astype(np.int64))
        labels = ids_t.clone()
        labels[:prefix_len] = -100
        return {"input_ids": ids_t, "labels": labels}

def collate(batch):
    L = max(x["input_ids"].numel() for x in batch)
    ids = torch.full((len(batch), L), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = x["input_ids"].numel()
        ids[i, :n] = x["input_ids"]; labels[i, :n] = x["labels"]
    return {"input_ids": ids, "labels": labels}

train_ds = SummarizationDataset(train_examples)
val_ds = SummarizationDataset(val_examples)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate, num_workers=0)

lens = np.array([len(e[0]) for e in train_examples])
print(f"Train sequence length: mean={lens.mean():.0f} p95={np.percentile(lens, 95):.0f} max={lens.max()}")

Tokenizing (pure-Python BPE; this can take a few minutes) ...
  [train] tokenized 500/6677 (4s)
  [train] tokenized 1000/6677 (8s)
  [train] tokenized 1500/6677 (12s)
  [train] tokenized 2000/6677 (15s)
  [train] tokenized 2500/6677 (19s)
  [train] tokenized 3000/6677 (22s)
  [train] tokenized 3500/6677 (26s)
  [train] tokenized 4000/6677 (30s)
  [train] tokenized 4500/6677 (34s)
  [train] tokenized 5000/6677 (38s)
  [train] tokenized 5500/6677 (41s)
  [train] tokenized 6000/6677 (45s)
  [train] tokenized 6500/6677 (49s)
train: 6677 examples | source truncated: 6552 (98.1%)
val: 351 examples | source truncated: 343 (97.7%)
Train sequence length: mean=2175 p95=2200 max=2200


In [21]:
# 9. Loss helper and sanity checks
def lm_loss(model, input_ids, labels, reduction="mean"):
    # Next-token cross-entropy computed ONLY at supervised positions. The LM head is applied to
    # those positions alone (the ~1.5k prompt positions are skipped), which saves memory and time
    # and gives exactly the same value as masking with ignore_index.
    h = model.forward_hidden(input_ids)[:, :-1]
    tgt = labels[:, 1:]
    mask = tgt != -100
    n = int(mask.sum())
    if n == 0:
        return h.sum() * 0.0, 0
    logits = F.linear(h[mask], model.token_emb.weight)
    return F.cross_entropy(logits.float(), tgt[mask], reduction=reduction), n

# (a) the example layout is what we think it is
ids, plen = train_examples[0]
assert ids[0] == BOS_ID and ids[-1] == EOS_ID
assert list(ids[plen - len(SEP_IDS):plen]) == SEP_IDS, "separator missing before the target"
print("Prompt tail :", repr(tokenizer.decode(ids[max(0, plen - 60):plen])))
print("Target start:", repr(tokenizer.decode(ids[plen:plen + 60])))

# (b) the masked-head loss equals the standard full-logits ignore_index loss
model.eval()
b = collate([train_ds[0]])
x, y = b["input_ids"].to(DEVICE), b["labels"].to(DEVICE)
with torch.no_grad():
    fast, n_tok = lm_loss(model, x, y)
    full_logits = model(x)
    ref = F.cross_entropy(full_logits[:, :-1].reshape(-1, full_logits.size(-1)).float(),
                          y[:, 1:].reshape(-1), ignore_index=-100)
assert abs(float(fast) - float(ref)) < 1e-4, (float(fast), float(ref))
print(f"Loss check OK: {float(fast):.4f} (supervised tokens: {n_tok}, sequence length: {x.shape[1]})")

# (c) KV-cached decoding reproduces the full forward pass
with torch.no_grad():
    n_chk = min(200, x.shape[1])
    t = x[:, :n_chk]
    full_h = model.forward_hidden(t)
    h1, past = model.forward_hidden(t[:, :n_chk - 1], use_cache=True)
    h2, _ = model.forward_hidden(t[:, n_chk - 1:], past_kvs=past, use_cache=True)
cache_diff = (h2[:, -1] - full_h[:, -1]).abs().max().item()
assert cache_diff < 1e-3, f"KV-cache mismatch: {cache_diff}"
print(f"KV-cache check OK (max|diff|={cache_diff:.2e})")
del full_logits, full_h, h1, h2, past

Prompt tail : "against the appellant 's stand. We accordingly dismiss the appeals and uphold the decisions of the High Court. There would be no order for costs. V.P.R. Appeals dismissed.\n\nSUMMARY:\n"
Target start: 'The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell'
Loss check OK: 1.5668 (supervised tokens: 701, sequence length: 2200)
KV-cache check OK (max|diff|=5.48e-06)


In [22]:
# 10. Train LoRMA+ adapters only (resumable; checkpoints store adapters only, not the 99M base weights)
train_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(train_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

n_micro = math.ceil(len(train_ds) / MICRO_BATCH_SIZE)           # micro-batches per epoch
steps_per_epoch = math.ceil(n_micro / GRAD_ACCUM_STEPS)         # optimizer steps per epoch
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-4, (step + 1) / warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)

log_path = OUTPUT_DIR / "train_log.jsonl"
best_path = OUTPUT_DIR / "exp1_lorma_best.pt"
latest_path = OUTPUT_DIR / "exp1_lorma_latest.pt"

# Avoid silently loading adapters from an earlier run when RESUME=False.
if not RESUME:
    for stale in (best_path, latest_path, log_path):
        if stale.exists():
            stale.unlink()
elif not latest_path.exists():
    raise FileNotFoundError(
        "RESUME=True but the latest training checkpoint is missing. "
        "Set RESUME=False to start a clean run."
    )
LORMA_META = {"rank": LORMA_RANK, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT,
              "targets": list(LORMA_TARGETS), "variant": "LoRMA+"}

def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def log_event(rec):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")

@torch.no_grad()
def evaluate_loss(max_examples=EVAL_MAX_EXAMPLES):
    # Token-weighted mean validation loss over a fixed subset.
    was_training = model.training
    model.eval()
    total, count = 0.0, 0
    for j, b in enumerate(val_loader):
        if j >= max_examples:
            break
        x, y = b["input_ids"].to(DEVICE), b["labels"].to(DEVICE)
        with amp_ctx():
            loss_sum, n = lm_loss(model, x, y, reduction="sum")
        if n and torch.isfinite(loss_sum):
            total += float(loss_sum); count += n
    model.train(was_training)
    return total / count if count else float("inf")

global_step, best_val, baseline_val = 0, float("inf"), None
if RESUME and latest_path.exists():
    st = load_torch(latest_path)
    if st["lorma"] != LORMA_META:
        raise RuntimeError(f"Resume checkpoint used different LoRMA settings: {st['lorma']} vs {LORMA_META}")
    load_lorma_state_dict(model, st["adapters"])
    optimizer.load_state_dict(st["optimizer"]); scheduler.load_state_dict(st["scheduler"])
    scaler.load_state_dict(st["scaler"])
    global_step, best_val, baseline_val = st["global_step"], st["best_val"], st.get("baseline_val")
    print(f"Resumed from step {global_step} (best_val={best_val:.4f})")
    del st
elif RESUME:
    print("RESUME=True but no latest checkpoint found; starting from scratch.")

if global_step == 0 and baseline_val is None:
    baseline_val = evaluate_loss()      # adapters are an exact identity here => this is the Phase 1 model
    print(f"Baseline validation loss (frozen Phase 1 model): {baseline_val:.4f}")
    log_event({"event": "baseline", "val_loss": baseline_val})

def checkpoint(val_loss=None, save_latest=True):
    global best_val
    if val_loss is not None and val_loss < best_val:
        best_val = val_loss
        atomic_save({"adapters": lorma_state_dict(model), "config": cfg, "lorma": LORMA_META,
                     "global_step": global_step, "val_loss": val_loss,
                     "base_checkpoint": str(CHECKPOINT_PATH)}, best_path)
        print(f"  saved BEST (val_loss={best_val:.4f})", flush=True)
    if save_latest:
        atomic_save({"adapters": lorma_state_dict(model), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                     "global_step": global_step, "best_val": best_val, "baseline_val": baseline_val,
                     "val_loss": val_loss, "config": cfg, "lorma": LORMA_META}, latest_path)

t_start = time.time()
last_eval_step = global_step
win_loss, win_n, n_nonfinite = 0.0, 0, 0
model.train()
for epoch in range(global_step // steps_per_epoch, NUM_EPOCHS):
    # Deterministic per-epoch order => a resumed run sees exactly the data it would have seen.
    order = torch.randperm(len(train_ds), generator=torch.Generator().manual_seed(SEED + epoch)).tolist()
    start_micro = (global_step - epoch * steps_per_epoch) * GRAD_ACCUM_STEPS
    loader = DataLoader(train_ds, batch_size=MICRO_BATCH_SIZE, sampler=order[start_micro * MICRO_BATCH_SIZE:],
                        collate_fn=collate, num_workers=0, pin_memory=USE_CUDA)
    optimizer.zero_grad(set_to_none=True)
    for it, b in enumerate(loader, start=start_micro):
        group_size = min(GRAD_ACCUM_STEPS, n_micro - (it // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS)
        x = b["input_ids"].to(DEVICE, non_blocking=True)
        y = b["labels"].to(DEVICE, non_blocking=True)
        with amp_ctx():
            loss, _ = lm_loss(model, x, y)
        if torch.isfinite(loss):
            scaler.scale(loss / group_size).backward()
            win_loss += float(loss); win_n += 1
        else:
            n_nonfinite += 1

        if (it + 1) % GRAD_ACCUM_STEPS == 0 or (it + 1) == n_micro:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(train_params, MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step(); global_step += 1

            if global_step % LOG_EVERY == 0:
                rec = {"epoch": epoch + 1, "step": global_step, "train_loss": win_loss / max(1, win_n),
                       "grad_norm": float(grad_norm), "lr": scheduler.get_last_lr()[0],
                       "nonfinite_batches": n_nonfinite, "elapsed_min": (time.time() - t_start) / 60}
                if USE_CUDA:
                    rec["max_mem_gb"] = torch.cuda.max_memory_allocated() / 2**30
                log_event(rec); print(rec, flush=True)
                win_loss, win_n = 0.0, 0
            do_eval = global_step % EVAL_EVERY == 0
            do_save = global_step % SAVE_EVERY == 0
            if do_eval or do_save:
                v = evaluate_loss() if do_eval else None
                if do_eval:
                    last_eval_step = global_step
                    log_event({"event": "eval", "step": global_step, "val_loss": v})
                    print(f"step={global_step} val_loss={v:.4f} (best={min(best_val, v):.4f})", flush=True)
                checkpoint(v, save_latest=True)
                model.train()

    if last_eval_step != global_step:       # end-of-epoch evaluation (skipped if we just evaluated)
        v = evaluate_loss(); last_eval_step = global_step
        log_event({"event": "epoch_end", "epoch": epoch + 1, "step": global_step, "val_loss": v})
        checkpoint(v, save_latest=True)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} complete | step={global_step} | val_loss={v:.4f} | best={best_val:.4f}")
        model.train()
    else:
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} complete | step={global_step} | best={best_val:.4f}")

if not best_path.exists():          # e.g. nothing improved because every evaluation was non-finite
    checkpoint(evaluate_loss(), save_latest=True)
training_summary = {"baseline_val_loss": baseline_val, "best_val_loss": float(best_val),
                    "global_step": global_step, "train_minutes": (time.time() - t_start) / 60}
print("Training complete:", training_summary, "\nBest adapters:", best_path)

Baseline validation loss (frozen Phase 1 model): 1.5834


/tmp/ipykernel_48/366589531.py:114: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  win_loss += float(loss); win_n += 1


{'epoch': 1, 'step': 50, 'train_loss': 1.4830920240283012, 'grad_norm': 0.3130607306957245, 'lr': 0.00012289156626506023, 'nonfinite_batches': 0, 'elapsed_min': 1.3317251801490784, 'max_mem_gb': 2.288969039916992}
{'epoch': 1, 'step': 100, 'train_loss': 1.3502300837635994, 'grad_norm': 0.33251291513442993, 'lr': 0.00019994904166131195, 'nonfinite_batches': 0, 'elapsed_min': 2.720869727929433, 'max_mem_gb': 2.288969039916992}
{'epoch': 1, 'step': 150, 'train_loss': 1.3214649733901025, 'grad_norm': 0.30846473574638367, 'lr': 0.00019920955570209284, 'nonfinite_batches': 0, 'elapsed_min': 4.106831125418345, 'max_mem_gb': 2.288969039916992}
{'epoch': 1, 'step': 200, 'train_loss': 1.3125318112969397, 'grad_norm': 0.2852948009967804, 'lr': 0.00019759681151020815, 'nonfinite_batches': 0, 'elapsed_min': 5.481094745794932, 'max_mem_gb': 2.288969039916992}
{'epoch': 1, 'step': 250, 'train_loss': 1.3105510449409485, 'grad_norm': 0.35669371485710144, 'lr': 0.00019512659598350376, 'nonfinite_batches

In [23]:
# 11. Restore the best adapters
best_obj = load_torch(best_path)
if best_obj["lorma"] != LORMA_META:
    raise RuntimeError("Best checkpoint was trained with different LoRMA settings than the current configuration.")
load_lorma_state_dict(model, best_obj["adapters"])
model.to(DEVICE).eval()
print(f"Restored best adapters | step={best_obj['global_step']} | val_loss={best_obj['val_loss']:.4f}")
del best_obj

Restored best adapters | step=1670 | val_loss=1.2689


In [24]:
# 12. Generation (greedy, KV-cached) and validation metrics
BLOCKED_IDS = [PAD_ID, tokenizer.unk_token_id, BOS_ID]

def build_inference_prompt(document, max_new_tokens):
    # Same prompt layout as training; the source window shrinks (never the prompt) to leave room to generate.
    budget = min(MAX_SOURCE_TOKENS, MAX_SEQ_LEN - PREFIX_OVERHEAD - max_new_tokens)
    if budget < 1:
        raise ValueError("max_new_tokens leaves no room for the source document")
    src = head_tail(tokenizer.encode(document), MAX_SOURCE_TOKENS)   # same nested window as training
    return build_prompt_ids(src, budget)

@torch.inference_mode()
def generate_summary(document, max_new_tokens=GEN_MAX_NEW_TOKENS, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE):
    model.eval()
    prompt = build_inference_prompt(document, max_new_tokens)
    max_new = min(max_new_tokens, model.max_positions - len(prompt))
    emb_w = model.token_emb.weight
    with amp_ctx():
        h, past = model.forward_hidden(torch.tensor([prompt], device=DEVICE), use_cache=True)
    generated, seen = [], {}
    n = no_repeat_ngram_size
    for _ in range(max_new):
        with amp_ctx():
            logits = F.linear(h[:, -1], emb_w)
        logits = logits.float()
        logits[:, BLOCKED_IDS] = -float("inf")
        key = None
        if n > 0 and len(generated) >= n - 1:
            key = tuple(generated[len(generated) - (n - 1):])     # the previous n-1 tokens
            banned = seen.get(key)
            if banned:
                logits[0, list(banned)] = -float("inf")
        nxt = int(torch.argmax(logits, dim=-1).item())
        if nxt == EOS_ID:
            break
        if key is not None:
            seen.setdefault(key, set()).add(nxt)
        generated.append(nxt)
        with amp_ctx():
            h, past = model.forward_hidden(torch.tensor([[nxt]], device=DEVICE), past_kvs=past, use_cache=True)
    return tokenizer.decode(generated).strip()

def generate_many(rows, label, every=10):
    preds, t0 = [], time.time()
    for i, r in enumerate(rows):
        preds.append(generate_summary(r["document"]))
        if (i + 1) % every == 0 or (i + 1) == len(rows):
            rate = (time.time() - t0) / (i + 1)
            print(f"[{label}] {i + 1}/{len(rows)} | {rate:.1f}s/doc | ETA {rate * (len(rows) - i - 1) / 60:.1f} min", flush=True)
    return preds

def compute_metrics(preds, refs, use_bertscore=True):
    metrics = {}
    try:
        ensure_package("rouge_score", "rouge-score")
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
        for k in ("rouge1", "rouge2", "rougeL"):
            metrics[f"{k}_f1"] = float(np.mean([s[k].fmeasure for s in scores]))
    except Exception as e:
        print("ROUGE skipped:", repr(e))
    if use_bertscore:
        try:
            ensure_package("bert_score", "bert-score")
            from bert_score import score as bert_score
            gc.collect()
            if USE_CUDA:
                torch.cuda.empty_cache()
            safe_preds = [p if p.strip() else "." for p in preds]      # empty strings break BERTScore
            _, _, f1 = bert_score(safe_preds, refs, lang="en", batch_size=BERTSCORE_BATCH_SIZE,
                                  device=str(DEVICE), verbose=False)
            metrics["bertscore_f1"] = float(f1.mean())
        except Exception as e:
            print("BERTScore skipped (needs the scoring model: internet or a cached copy):", repr(e))
    return metrics

val_subset = val_rows[:min(VAL_SCORE_EXAMPLES, len(val_rows))]
val_preds = generate_many(val_subset, "val")
val_refs = [r["summary"] for r in val_subset]

val_metrics = compute_metrics(val_preds, val_refs)
val_metrics["empty_predictions"] = int(sum(not p.strip() for p in val_preds))
val_metrics["mean_pred_words"] = float(np.mean([len(p.split()) for p in val_preds]))
val_metrics["mean_ref_words"] = float(np.mean([len(p.split()) for p in val_refs]))
print(json.dumps(val_metrics, indent=2))
print("\nExample prediction:\n", val_preds[0][:600])

[val] 10/100 | 24.8s/doc | ETA 37.2 min
[val] 20/100 | 24.4s/doc | ETA 32.5 min
[val] 30/100 | 24.4s/doc | ETA 28.4 min
[val] 40/100 | 24.4s/doc | ETA 24.4 min
[val] 50/100 | 24.5s/doc | ETA 20.4 min
[val] 60/100 | 24.5s/doc | ETA 16.3 min
[val] 70/100 | 24.5s/doc | ETA 12.2 min
[val] 80/100 | 24.5s/doc | ETA 8.2 min
[val] 90/100 | 24.5s/doc | ETA 4.1 min
[val] 100/100 | 24.5s/doc | ETA 0.0 min


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{
  "rouge1_f1": 0.2751417780402789,
  "rouge2_f1": 0.08822961416081052,
  "rougeL_f1": 0.1981684192387962,
  "bertscore_f1": 0.7855862975120544,
  "empty_predictions": 0,
  "mean_pred_words": 334.25,
  "mean_ref_words": 952.83
}

Example prediction:
 The appellants and respondent No. 4 were the daughters and legal representatives of Savoy Ranganna who were the daughters of Savoy Ranganna who was the brother of the appellant. The suit was filed by the appellants and respondent No. 4 as the brother of the appellant. The suit was filed by the appellants and respondent No. 4 as the brother of the appellant. The suit was filed by the appellants and respondent No. 4 as the brother of the appellant. The suit was filed by the appellants and respondent No. 4 as the brother of the appellant. The suit was filed by the appellants and respondent No. 4 


In [25]:
# 13. Save validation predictions, metrics and experiment metadata
with open(OUTPUT_DIR / "validation_predictions.json", "w", encoding="utf-8") as f:
    json.dump([{"id": r["id"], "summary": p, "reference": r["summary"]} for r, p in zip(val_subset, val_preds)],
              f, ensure_ascii=False, indent=2)

metadata = {
    "experiment": "Phase6-Exp1-Model4-LoRMA+",
    "base_architecture": "efficient_gqa_rope_swiglu_decoder",
    "checkpoint": str(CHECKPOINT_PATH), "tokenizer": str(TOKENIZER_PATH),
    "lorma": LORMA_META, "trainable_parameters": n_trainable, "total_parameters": n_total,
    "max_source_tokens": MAX_SOURCE_TOKENS, "max_target_tokens": MAX_TARGET_TOKENS, "max_seq_len": MAX_SEQ_LEN,
    "micro_batch_size": MICRO_BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS, "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
    "amp_dtype": str(AMP_DTYPE) if USE_CUDA else "none",
    "gen_max_new_tokens": GEN_MAX_NEW_TOKENS, "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    "train_examples": len(train_rows), "validation_examples": len(val_rows),
    "validation_scored_examples": len(val_subset),
    "training": training_summary, "validation_metrics": val_metrics,
    "seed": SEED, "torch_version": torch.__version__,
}
with open(OUTPUT_DIR / "experiment_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

{
  "experiment": "Phase6-Exp1-Model4-LoRMA+",
  "base_architecture": "efficient_gqa_rope_swiglu_decoder",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt",
  "tokenizer": "/kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json",
  "lorma": {
    "rank": 8,
    "alpha": 16,
    "dropout": 0.05,
    "targets": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "variant": "LoRMA+"
  },
  "trainable_parameters": 1705984,
  "total_parameters": 101015424,
  "max_source_tokens": 1500,
  "max_target_tokens": 700,
  "max_seq_len": 2200,
  "micro_batch_size": 1,
  "grad_accum_steps": 8,
  "epochs": 2,
  "learning_rate": 0.0002,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "amp_dtype": "torch.float16",
  "gen_max_new_tokens": 700,
  "no_repeat_ngram_size": 0,
  "train_examples": 6677,
  "validation_examples": 351,
  "validation_scored_examples": 100,
  "training": {
  

In [26]:
# 13A. Exact Phase 6 evaluator embedded in the notebook (no separate .py file needed)
# This reproduces the supplied evaluate_phase6.py logic and runs it directly on validation outputs.
# Dependencies: rouge-score, bert-score, torch.

ensure_package("rouge_score", "rouge-score")
ensure_package("bert_score", "bert-score")
from rouge_score import tokenizers, rouge_scorer
import bert_score

def evaluator_load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def evaluator_lcs(a, b):
    # Bit-parallel LCS implementation from the supplied evaluator.
    if len(a) > len(b):
        a, b = b, a
    masks = {}
    for i, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << i)
    state = 0
    for token in b:
        mask = masks.get(token, 0)
        x = mask | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")

def evaluator_read_predictions(value):
    if isinstance(value, dict):
        output = value
    elif isinstance(value, list):
        output = {}
        for record in value:
            if not isinstance(record, dict) or "id" not in record or "summary" not in record:
                raise ValueError("Each prediction item must contain id and summary.")
            if record["id"] in output:
                raise ValueError(f"Duplicate prediction ID: {record['id']}")
            output[record["id"]] = record["summary"]
    else:
        raise ValueError("Prediction must be a JSON object or list of objects.")

    for key, summary in output.items():
        if not isinstance(key, str) or not isinstance(summary, str) or not summary.strip():
            raise ValueError(f"Invalid summary for ID {key!r}.")
    return output

def evaluate_phase6_exact(truth_file, prediction_file):
    truth = evaluator_load(truth_file)
    prediction_map = evaluator_read_predictions(evaluator_load(prediction_file))

    if not isinstance(truth, dict) or not truth:
        raise ValueError("Truth must be a non-empty JSON object.")
    if set(truth) != set(prediction_map):
        raise ValueError(
            f"ID mismatch: missing={len(set(truth)-set(prediction_map))}, "
            f"extra={len(set(prediction_map)-set(truth))}"
        )

    predictions = [prediction_map[key].strip() for key in truth]
    references = [truth[key].strip() for key in truth]

    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    tokenizer = tokenizers.DefaultTokenizer(use_stemmer=True)
    rouge1 = rouge_l = 0.0

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1 += scores["rouge1"].fmeasure

        pred_tokens = tokenizer.tokenize(pred)
        ref_tokens = tokenizer.tokenize(ref)
        if not pred_tokens and not ref_tokens:
            rouge_l_doc = 1.0
        elif not pred_tokens or not ref_tokens:
            rouge_l_doc = 0.0
        else:
            common = evaluator_lcs(pred_tokens, ref_tokens)
            precision = common / len(pred_tokens)
            recall = common / len(ref_tokens)
            rouge_l_doc = (
                0.0 if precision + recall == 0
                else 2 * precision * recall / (precision + recall)
            )
        rouge_l += rouge_l_doc

    count = len(truth)
    rouge1 /= count
    rouge_l /= count

    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, bert_f1 = bert_score.score(
        predictions, references, lang="en", batch_size=8,
        verbose=False, device=device
    )
    bert_f1_mean = float(bert_f1.mean().item())

    print(f"Documents: {count}")
    print(f"BERTScore F1: {bert_f1_mean:.6f}")
    print(f"ROUGE-1: {rouge1:.6f}")
    print(f"ROUGE-L: {rouge_l:.6f}")

    return {
        "documents": count,
        "bertscore_f1": bert_f1_mean,
        "rouge1": rouge1,
        "rougeL": rouge_l,
    }

# Save files in precisely the formats expected by the original evaluator.
val_truth = {str(record["id"]): record["summary"] for record in val_subset}
truth_path = OUTPUT_DIR / "validation_truth.json"
evaluator_prediction_path = OUTPUT_DIR / "validation_predictions_for_evaluator.json"

with open(truth_path, "w", encoding="utf-8") as f:
    json.dump(val_truth, f, ensure_ascii=False, indent=2)
with open(evaluator_prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        [{"id": str(record["id"]), "summary": pred}
         for record, pred in zip(val_subset, val_preds)],
        f, ensure_ascii=False, indent=2
    )

print("Running embedded version of your evaluate_phase6.py")
official_validation_metrics = evaluate_phase6_exact(
    truth_path, evaluator_prediction_path
)


Running embedded version of your evaluate_phase6.py


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Documents: 100
BERTScore F1: 0.785586
ROUGE-1: 0.275142
ROUGE-L: 0.198168


In [27]:
# 14. Official test predictions -> [{"id": "...", "summary": "..."}]
# Resumable: every prediction is appended to a .jsonl file, so an interrupted run continues where it stopped.
FALLBACK_SUMMARY = "The judgment could not be summarized from the supplied context."

if not RUN_TEST_INFERENCE:
    print("RUN_TEST_INFERENCE=False; skipping test predictions.")
else:
    partial_path = OUTPUT_DIR / "submission_partial.jsonl"
    done = {}
    if partial_path.exists() and RESUME_TEST_PREDICTIONS:
        with open(partial_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    if isinstance(rec.get("id"), str) and isinstance(rec.get("summary"), str) and rec["summary"].strip():
                        done[rec["id"]] = rec["summary"]
        print(f"Found {len(done)} cached test predictions; continuing.")
    t0, n_new = time.time(), 0
    file_mode = "a" if (partial_path.exists() and RESUME_TEST_PREDICTIONS) else "w"
    with open(partial_path, file_mode, encoding="utf-8") as f:
        for i, r in enumerate(test_records):
            rid = str(r["id"])
            if rid in done:
                continue
            summary = generate_summary(r["document"]) or FALLBACK_SUMMARY
            done[rid] = summary; n_new += 1
            f.write(json.dumps({"id": rid, "summary": summary}, ensure_ascii=False) + "\n"); f.flush()
            if n_new % 10 == 0:
                rate = (time.time() - t0) / n_new
                print(f"Test {len(done)}/{len(test_records)} | {rate:.1f}s/doc | "
                      f"ETA {rate * (len(test_records) - len(done)) / 60:.1f} min", flush=True)

    test_preds = [{"id": str(r["id"]), "summary": done[str(r["id"])]} for r in test_records]
    assert len(test_preds) == len(test_records)
    assert len({x["id"] for x in test_preds}) == len(test_preds)
    assert all(x["summary"].strip() for x in test_preds)
    # Competition archive: prediction.json must be directly at the ZIP root.
    pred_path = OUTPUT_DIR / "prediction.json"
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(test_preds, f, ensure_ascii=False, indent=2)

    import zipfile
    zip_path = OUTPUT_DIR / "phase6_exp1_submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    # Verify archive structure, IDs, and non-empty summaries before upload.
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        assert names == ["prediction.json"], f"Unexpected ZIP contents: {names}"
        archived = json.loads(zf.read("prediction.json").decode("utf-8"))
        assert len(archived) == len(test_records)
        assert [x["id"] for x in archived] == [str(r["id"]) for r in test_records]
        assert all(isinstance(x["summary"], str) and x["summary"].strip() for x in archived)

    print("JSON:", pred_path, "| records:", len(test_preds))
    print("ZIP :", zip_path, "| contents:", names)


Test 10/100 | 24.8s/doc | ETA 37.3 min
Test 20/100 | 24.8s/doc | ETA 33.0 min
Test 30/100 | 24.7s/doc | ETA 28.9 min
Test 40/100 | 24.4s/doc | ETA 24.4 min
Test 50/100 | 24.5s/doc | ETA 20.4 min
Test 60/100 | 24.6s/doc | ETA 16.4 min
Test 70/100 | 24.6s/doc | ETA 12.3 min
Test 80/100 | 24.6s/doc | ETA 8.2 min
Test 90/100 | 24.6s/doc | ETA 4.1 min
Test 100/100 | 24.6s/doc | ETA 0.0 min
JSON: /kaggle/working/phase6_exp1_model4_lorma/prediction.json | records: 100
ZIP : /kaggle/working/phase6_exp1_model4_lorma/phase6_exp1_submission.zip | contents: ['prediction.json']


## Experiment 1 notes / limitations

- This is **LoRMA+**, the additive rank-inflation variant of Low-Rank Multiplicative Adaptation. Its transformation is $(I + \tfrac{\alpha}{r}BA)\,W_0 x$, which equals the frozen layer at initialisation because `B` starts at zero (the notebook asserts this numerically).
- The model is a causal decoder. Training is teacher-forced next-token prediction with loss only on the summary and EOS.
- This first run uses a bounded source window (opening 2/3 + closing 1/3) and target truncation. It is a baseline long-document policy, not a claim that the entire judgment is consumed. When prompt + source + target would exceed `MAX_SEQ_LEN`, the *source* window shrinks; the instruction and the `SUMMARY:` separator are never cut.
- Validation documents are grouped by normalised document hash to reduce duplicate leakage. For final reporting record: the split seed, number of duplicate groups, baseline and best validation loss, ROUGE-1/2/L F1 and BERTScore F1 (all are written to `experiment_metadata.json`).
- BERTScore truncates long inputs at the scoring model's own length limit (512 tokens), so very long summaries are only partly scored.
- Adapter checkpoints are small (adapters only). To reuse them later, rebuild the base model from the Phase 1 checkpoint, call `attach_lorma`, then `load_lorma_state_dict`.
- If GPU memory is tight: lower `MAX_SOURCE_TOKENS` to 1000 and `MAX_TARGET_TOKENS` to 400, or keep `MICRO_BATCH_SIZE=1` and raise `GRAD_ACCUM_STEPS`.
- If generated summaries repeat themselves, set `NO_REPEAT_NGRAM_SIZE` to roughly 10–16 and compare validation metrics.
- Do not train on the official test set or use its outputs for hyperparameter selection.
- KV-cached greedy decoding reduces repeated full-prefix computation during generation.
- Training computes the LM head only at supervised target positions; a sanity check compares this masked loss against standard full-logit loss.
- Fresh training runs clear stale adapter checkpoints, preventing an old best checkpoint from being mistaken for the current run.
- The final test cell writes the competition-required `prediction.json` and a ZIP with that file directly at its root, then validates IDs and non-empty summaries.
- The complete supplied `evaluate_phase6.py` scoring logic is embedded in cell 13A and is executed directly on the held-out validation predictions; no external evaluator file or `EVALUATOR_PATH` setting is needed.
\n- Checkpoint config loading now recognizes common RMSNorm epsilon aliases (`rms_norm_eps`, `norm_eps`, `layer_norm_epsilon`, `eps`). If absent, it warns and uses `1e-5`, the RMSNorm class default; verify this matches the original Phase 1 model if its implementation used a custom epsilon.\n